# Zou Lab Control：离线跑一次正式虚拟链

这一本教程只使用安装后的单一 ZLC distribution。它在临时 workspace 中启动 virtual apparatus，完成 Camera Measurement → Runtime publication → Notebook plot，并在最后关闭全部 owner。

In [ ]:
from importlib.metadata import version
from pathlib import Path
import zou_lab_control
import zlc_workbench

print('product version:', version('zou-lab-control'))
print('product bootstrap:', Path(zou_lab_control.__file__).resolve())
print('workbench layer:', Path(zlc_workbench.__file__).resolve())

## 1. 临时 workspace 与 virtual apparatus

`template='virtual'`走与 Device Manager 相同的 installation contract。正式 imaging pulse 从 wheel 的 package-data 种入 workspace，不从 checkout 或 tests 复制。

In [ ]:
from tempfile import TemporaryDirectory
from zlc_workbench.session import ExperimentSession, Workspace

temporary = TemporaryDirectory()
workspace = Workspace(Path(temporary.name)).prepare()
session = ExperimentSession.open(workspace, template='virtual')
assert session.failures == {}
pulse = session.load_pulse('imaging_template')
print('workspace:', workspace.root)
print('pulse:', pulse['name'])

## 2. 一次 canonical Camera Measurement

Measurement只负责arm/collect；`ExperimentSession.fire`负责时序器。返回的 publication 与 SignalDataPlane 中的 front 是同一 revision。

In [ ]:
import numpy as np
from zlc_atom.nodes.camera_measurement import (
    CameraMeasurementNode, CameraMeasurementRequest,
)

node = CameraMeasurementNode(
    camera=session.camera,
    request=CameraMeasurementRequest('camera', 0.005, None, 1, 3),
    signal_plane=session.signal_plane,
    producer='notebook',
)
capture = node.prepare()
session.fire(shots=1)
result = capture.collect()
signal = node.signal_key('frames')
snapshot = result.publication.value(signal).snapshot
front_snapshot = session.signal_plane.freeze().value(signal).snapshot
assert front_snapshot.ref == snapshot.ref
print('revision:', snapshot.ref.revision.value)
print('frame tensor:', np.asarray(snapshot.block.values).shape)

## 3. 同一 snapshot 的 NotebookView

NotebookView复用正式 RasterPlotHost；这里不创建第二份数据或 pyplot 状态。

In [ ]:
import zlc_plot as plot

y_axis, x_axis = snapshot.block.schema.cell_schema.data_axes[-2:]
plot_session = plot.image(
    snapshot,
    plot.AxisRef.data(x_axis.axis_id.value),
    plot.AxisRef.data(y_axis.axis_id.value),
    labels=plot.PlotLabels(title='Virtual camera frame', x='X', y='Y'),
)
view = plot.show(plot_session, close_session_on_close=True)
assert view.describe_display().kind.value == 'image'
print('display revision:', plot_session.data_revision)

## 4. 保存边界与收工

正式 Figure 保存由 Task Console 的 Panel Edit 完成，因为只有 Panel 同时知道 exact revision、axis fate、fit、overlay 与 viewport。教程最后显式关闭 view、plot session、device installation 与 signal plane。

In [ ]:
view.close()
session.close()
temporary.cleanup()
print('all tutorial owners closed')